In [1]:
import matplotlib
matplotlib.use("Agg")
import numpy as np
import matplotlib.pyplot as plt
import h5py

import importlib
import os
import sys
import re
import glob

In [2]:
path_openket = "//home//ultrxvioletx//GitHub//openket"
if path_openket not in sys.path:
    sys.path.append(path_openket)
# elimina el caché de func.py para que no lea los datos anteriores
if 'func' in sys.modules:
    del sys.modules['func']
if os.path.exists('func.py'):
    os.remove('func.py')

from openket.core.diracobject import Ket, Bra, Operator, AnnihilationOperator, CreationOperator
from openket.core.evolution import build_ode, sym2num, init_state
from openket.core.metrics import dag, comm, ptrace, trace, normalize, sub_qexpr, op2dict, qmatrix

import style
importlib.reload(style)
from style import set_style, colores, format_ax
set_style()

2026-03-18 21:41:27,733 - openket - INFO - openket v0.1.0 initialized successfully.


##### funciones auxiliares

In [3]:
def convergencia(tiempo, fotones, tol=1e-3, porcentaje=0.1):
    npoints = int(len(tiempo) * porcentaje)
    if npoints < 2:
        return False, np.nan # no hay suficientes puntos
    fotones = fotones[-npoints:]
    tiempo = tiempo[-npoints:]
    df = np.gradient(fotones, tiempo)
    df_mean = np.mean(np.abs(df))
    return df_mean < tol, df_mean

In [4]:
def procesafile(at, lvl, folder):
    global detunings, Nss, non_converged, Probs, niveles
    detunings = []
    Probs = [[] for _ in np.arange(lvl**at)]
    Nss = []
    
    non_converged = []
    files = glob.glob(os.path.join(path,'*.h5'))
    for file in files:
        with h5py.File(file, 'r') as f:
            dataset = os.path.basename(file).replace('.h5', '')
            
            detuning = f[dataset].attrs[f'detuning_bc']
            tt = f[dataset].attrs['t']
            niveles = f[dataset].attrs['probs_label']
            tiempo = np.linspace(float(tt[0]), float(tt[1]), int(tt[2]))
    
            if dataset not in f:
                print(f"  no se encontró el dataset '{dataset}' en el archivo. Saltando.")
                continue

            detunings.append(detuning)
            rho = f[dataset][:]
            lvl_prob = [rho[:, i] for i in np.arange(lvl**at)] # las primeras n columnas son las probabilidades
            N_expect = rho[:, lvl**at] # la última columna es el valor medio de fotones en la cavidad
            N_expect = np.real(N_expect)

            # promediamos el último 25% de los puntos
            npoints = int(rho.shape[0] * 0.25)
            Nss.append(np.mean(N_expect[-npoints:]))
            for i,ele in enumerate(lvl_prob):
                Probs[i].append(np.mean(ele[-npoints:]))
            
            is_converged, derivative = convergencia(tiempo, N_expect)
            if not is_converged:
                non_converged.append([detuning, np.mean(N_expect[-npoints:])])
                
    # ordena por detuning para mejorar la gráfica
    detunings = np.array(detunings)
    sorti = np.argsort(detunings)
    
    detunings = detunings[sorti]
    Nss = np.array(Nss)[sorti]
    Probs = [np.real(np.array(p))[sorti] for p in Probs]

##### 1 átomo 4 niveles

In [5]:
at,lvl = 1,4
path = os.path.join("detunings",f"{at}at{lvl}lvl","EIT")

procesafile(at, lvl, path)

plt.plot(detunings, Nss, "s-", markersize=5, linewidth=1, color=colores[2])
if non_converged:
    dtn = [ele[0] for ele in non_converged]
    nmean = [ele[1] for ele in non_converged]
    plt.scatter(dtn, nmean, color="red")
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.xlabel(r"$\Delta$")
plt.ylabel(r"Número medio de fotones $\langle N \rangle$")
ax = plt.gca()
format_ax(ax, xstep=1, ystep=0.004, ymax=max(Nss)*1.1)
plt.savefig(f"../figuras/fig:deltas_1at4lvl_fotones.png")
plt.close()

In [6]:
plvls = ["e", "s"]
for j,plvl in enumerate(plvls):
    pattern_str = plvl.replace("_", ".")
    pattern_str = pattern_str[:at] # recorta al largo del dataset
    pattern = re.compile(f"^{pattern_str}$")
    indices = [i for i, lbl in enumerate(niveles) if pattern.search(lbl)]
    prob = np.sum([Probs[i] for i in indices], axis=0)
    
    plt.plot(detunings, prob, "s-", label=rf"$P_{plvl}$", markersize=5, linewidth=1, color=colores[j])
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.xlabel(r"$\Delta$")
plt.ylabel(r"Probabilidad de excitación")
ax = plt.gca()
format_ax(ax, xstep=1, ystep=0.01)
plt.legend()
plt.savefig(f"../figuras/fig:deltas_1at4lvl_excitado.png")
plt.close()